In [ ]:
import operator
import os
import re
from typing import Annotated, List, Literal

import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send

load_dotenv()  

In [ ]:
### llm setup
llm = ChatGroq(model="openai/gpt-oss-120b", max_retries=8)  # retries ride out Groq free-tier rate limits
llm.invoke("hi").content

In [ ]:
### Retrievers from first to end
urls=[
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2004.04906v3.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2005.11401v4.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2212.10496v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2212.10511v4.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2305.06983v2.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2310.11511v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2312.10997v5.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2401.15884v3.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2401.18059v1.pdf",
r"D:\d\NLP\Adaptive_Jev_RAG\pdf\2403.14403v2.pdf"
]

pdfs =[]
for pdf in urls:
    load = PyPDFLoader(pdf).load()
    pdfs.extend(load)
pdfs

In [ ]:
embeddings = HuggingFaceEmbeddings(model_kwargs={"device": "cpu"})  # keep the GPU for the router model

In [ ]:
def VS_creation(whole):
    splitter = RecursiveCharacterTextSplitter(chunk_size =1000,chunk_overlap=150)
    chunks = splitter.split_documents(whole)
    # start from an empty collection so re-running this cell doesn't store every chunk again
    Chroma(collection_name="papers", embedding_function=embeddings).delete_collection()
    vector_s = Chroma.from_documents(chunks ,embeddings, collection_name="papers")
    return vector_s.as_retriever(search_kwargs={"k": 6})

In [ ]:
vs = VS_creation(pdfs)

In [ ]:
vs.invoke("what is RAG?")

In [ ]:
from langchain_core.documents import Document
ROUTES = ["direct_answer", "web_search", "rag_retrieval"]

class RAGState(BaseModel):
    question: str = ""
    messages: Annotated[List[AnyMessage], add_messages] = Field(default_factory=list)
    documents: List[Document] = Field(default_factory=list)  # or Annotated[..., operator.add] for parallel retrievers
    decision: Literal["direct_answer", "web_search", "rag_retrieval"] | None = None
    confidence: float = 0.0
    feedback: str = ""
    generation: str = ""
    retries: int = 0


In [ ]:
def retrieve_node(state:RAGState)->dict:
    question = state.question
    result = vs.invoke(question)
    return{"documents":result}

In [ ]:
retrieve_node(RAGState(question="what is the transformers architecture about"))


In [ ]:
GRADER_PROMPT = SystemMessage(content="""You are a relevance grader in an Adaptive RAG pipeline.
A router has already judged the user's QUESTION complex enough to need the internal knowledge base of research papers.
You will receive that QUESTION and one retrieved DOCUMENT chunk from the knowledge base.

Decide whether the document contains information that helps answer the question.
Your grades decide the next step: relevant chunks are passed to the answer generator,
and if too few chunks are relevant, the pipeline adapts (rewrites the query or falls back to web search).

Grade "Yes" if the chunk:
- directly answers the question, or part of it, or
- contains facts, definitions, methods, or results that are clearly about the question's topic.

Grade "No" if the chunk:
- is only loosely related (shares a keyword but not the meaning),
- is a reference list, author list, acknowledgements, table of contents, or other boilerplate,
- is about a different concept than the one asked about.

Be strict: the goal is to filter out irrelevant chunks, not to answer the question.
Judge only the given chunk; do not use outside knowledge to fill gaps.""")


In [ ]:
REWRITE_PROMPT = SystemMessage(content="""You are a query rewriter in an Adaptive RAG pipeline.
A router sent the user's QUESTION to the internal knowledge base of research papers, but too few retrieved chunks were relevant.
Your rewritten query will be used for the next attempt: either retrieving again from the knowledge base or searching the web.

Rewrite the question into a single, concise search query that is most likely to return relevant, factual results.

Guidelines:
- Keep the original intent and all key entities, names, acronyms, and technical terms.
- Expand ambiguous acronyms when it helps (e.g. "RAG" -> "retrieval-augmented generation").
- Use the vocabulary a research paper on the topic would use (method names, task names, standard terminology).
- Remove filler words, greetings, and conversational phrasing.
- Prefer keywords and specific terms over full sentences; about 5-15 words.
- Do not answer the question, add facts, or introduce assumptions not present in it.
- Output only the rewritten query, with no explanation or quotes.""")


In [ ]:
class RewrittenQuery(BaseModel):
    """Web-search-optimized version of the user's question."""
    query: str = Field(description="The rewritten web search query")

rewriter = llm.with_structured_output(RewrittenQuery).with_retry(stop_after_attempt=3)
def transform_query_node(state: RAGState) -> dict:
    result = rewriter.invoke([
        REWRITE_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}"),
    ])
    return {"question": result.query}


In [ ]:
GENERATION_PROMPT = SystemMessage(content="""You are the answer generator in an Adaptive RAG pipeline about retrieval-augmented generation and LLM research.
A router judged the user's QUESTION too complex to answer from model knowledge alone, so it was sent to retrieval.
You will receive the QUESTION and a set of CONTEXT passages. Each passage is either a chunk from a research paper in the internal knowledge base or a web search result, and each one is labeled with its source.

Answer the question using only the information in the context.

Guidelines:
- Base every claim on the context; do not add facts from your own knowledge.
- If the context only partly answers the question, answer that part and clearly say what is missing.
- If the context does not contain the answer, say "I could not find this in the provided sources." and do not guess.
- Cite the source of each key claim in square brackets, e.g. [2005.11401v4.pdf, p. 3] or [https://example.com].
- If sources disagree, point out the disagreement instead of picking one silently.
- Since the question was routed as complex, give a well-structured answer: open with a one-sentence answer (no heading), then the supporting analysis (comparisons, steps, trade-offs) as the question requires. Use short paragraphs or bullet points when that is easier to read.
- Explain technical terms briefly when they are central to the answer.""")

In [ ]:
class GradeDocument(BaseModel):
    """Relevance grade for one chunk retrieved from the research-paper knowledge base in the Adaptive RAG pipeline."""
    relevant: Literal["Yes", "No"] = Field(
        description="'Yes' if the chunk contains facts, definitions, methods, or results that help answer the question; "
                    "'No' if it is only loosely related, about a different concept, or boilerplate (references, author list, acknowledgements)."
    )

grader = llm.with_structured_output(GradeDocument).with_retry(stop_after_attempt=3)  # gpt-oss occasionally skips the tool call

def grade_document(state: RAGState) -> dict:
    relevant_docs = []
    for doc in state.documents:
        result = grader.invoke([
            GRADER_PROMPT,
            HumanMessage(content=f"QUESTION:\n{state.question}\n\nDOCUMENT:\n{doc.page_content}"),
        ])
        if result.relevant == "Yes":
            relevant_docs.append(doc)

    return {
        "documents": relevant_docs,
        "feedback": "Yes" if relevant_docs else "No",
    }

In [ ]:
## web search node
def web_search_node(state: RAGState) -> dict:
    try:
        results = DDGS().text(state.question, max_results=5)
    except Exception as e:
        print(f"Web search failed: {e}")
        return {}  

    web_docs = [
        Document(page_content=r["body"], metadata={"source": r["href"], "title": r["title"]})
        for r in results
    ]
    return {"documents": state.documents + web_docs}


In [ ]:
### generate node

def format_docs(docs: List[Document]) -> str:
    parts = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page")
        label = f"{source}, p. {page + 1}" if page is not None else source
        parts.append(f"[{label}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def generate_node(state: RAGState) -> dict:
    context = format_docs(state.documents)
    response = llm.invoke([
        GENERATION_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}\n\nCONTEXT:\n{context}"),
    ])
    return {"generation": response.content, "messages": [response]}


In [ ]:
def router_node(state:RAGState)-> Literal["transform_query","generate"]:
    if state.feedback =="Yes":
        return "generate"
    return "transform_query"

In [ ]:
graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve_node)
graph.add_node("grade_documents",grade_document)
graph.add_node("transform_query",transform_query_node)
graph.add_node("web_search",web_search_node)
graph.add_node("generate",generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve","grade_documents")
graph.add_conditional_edges("grade_documents",router_node)
graph.add_edge("transform_query","web_search")
graph.add_edge("web_search","generate")
graph.add_edge("generate",END)

graph_builder = graph.compile()

try:
    display(Image(graph_builder.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
except Exception:
    print(graph_builder.get_graph().draw_mermaid())  # mermaid.ink unreachable: show the diagram as text

In [ ]:
test_cases = [
    # In the PDFs: should go retrieve -> grade -> generate
    {"question": "What is the difference between the RAG-Sequence and RAG-Token models?", "expect": "local"},  # 2005.11401 RAG
    {"question": "What are reflection tokens in Self-RAG and what are they used for?", "expect": "local"},      # 2310.11511 Self-RAG

    # Not in the PDFs: grader should reject the chunks -> transform_query -> web_search -> generate
    {"question": "Who won the 2022 FIFA World Cup final?", "expect": "web"},
]

for case in test_cases:
    print("=" * 80)
    print("Q:", case["question"], f"(expected: {case['expect']})")
    path = []
    for update in graph_builder.stream({"question": case["question"]}, stream_mode="updates"):
        path.extend(update.keys())
        if "generate" in update:
            final = update["generate"]
    went_web = "web_search" in path
    print("Path:", " -> ".join(path))
    print("Went to web:", went_web, "-> OK" if went_web == (case["expect"] == "web") else "-> WRONG")
    print("Answer:", final["generation"][:500], "...")

#### WEB search normal one


In [ ]:
## web search node
def web_search_node(state: RAGState) -> dict:
    try:
        results = DDGS().text(state.question, max_results=5)
    except Exception as e:
        print(f"Web search failed: {e}")
        return {}  

    web_docs = [
        Document(page_content=r["body"], metadata={"source": r["href"], "title": r["title"]})
        for r in results
    ]
    return {"documents": state.documents + web_docs}


In [ ]:
SYSTEM_PROMPT = SystemMessage(content="""You are a helpful, knowledgeable assistant in an Adaptive RAG system specialised in retrieval-augmented generation and LLM research.
A router has judged the user's message simple enough to answer directly from your own knowledge, without retrieving documents or searching the web.

Guidelines:
- Answer clearly and concisely: give the direct answer first, then a short explanation only if it helps.
- For greetings and small talk, reply briefly and naturally, and offer help.
- For definitions and concepts, give an accurate, plain-language explanation; add a short example when it makes the idea clearer.
- For math or simple reasoning, show the key steps and the final result.
- Reply in the same language the user writes in.
- Be honest about uncertainty: if you are not sure, say so instead of guessing.
- Do not invent facts, numbers, paper titles, citations, or links.
- If the question actually needs up-to-date information (news, prices, recent releases) or detailed analysis of specific papers, say that you cannot answer it reliably from memory and that it needs a web search or the document knowledge base.""")
def llm_answer_node(state: RAGState) -> dict:
    response = llm.invoke([
        SYSTEM_PROMPT,
        HumanMessage(content=state.question),
    ])
    return {"generation": response.content, "messages": [response]}


### BERT router (replaces JEV)

Same pipeline as `adaptive.ipynb`; only the router changes. Instead of JEV (a 4B LLM read zero-shot), a
`bert-base-uncased` classifier is fine-tuned on the labeled routing queries in `eval/router_dataset.json` (train split).

In [ ]:
import json
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BERT_MODEL = "bert-base-uncased"
THRESHOLD = 0.7
ROUTES = ["direct_answer", "web_search", "rag_retrieval"]
device = "cuda" if torch.cuda.is_available() else "cpu"
router_train = json.loads(Path("eval/router_dataset.json").read_text(encoding="utf-8"))["train"]

torch.manual_seed(42)
start = time.perf_counter()
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL, num_labels=len(ROUTES), id2label=dict(enumerate(ROUTES)), label2id={r: i for i, r in enumerate(ROUTES)}
).to(device)
BERT_LOAD_S = time.perf_counter() - start


def collate(batch):
    enc = bert_tokenizer([b["query"] for b in batch], padding=True, truncation=True, max_length=64, return_tensors="pt")
    enc["labels"] = torch.tensor([ROUTES.index(b["label"]) for b in batch])
    return enc


EPOCHS = 8
loader = DataLoader(router_train, batch_size=16, shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(42))
optimizer = torch.optim.AdamW(bert.parameters(), lr=5e-5, weight_decay=0.01)

start = time.perf_counter()
bert.train()
for epoch in range(EPOCHS):
    total = 0.0
    for batch in loader:
        loss = bert(**{k: v.to(device) for k, v in batch.items()}).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total += loss.item()
    print(f"epoch {epoch + 1}: loss {total / len(loader):.4f}")
if torch.cuda.is_available():
    torch.cuda.synchronize()
BERT_TRAIN_S = time.perf_counter() - start
bert.eval()
print(f"fine-tuned on {len(router_train)} queries in {BERT_TRAIN_S:.1f}s")

In [ ]:
@torch.inference_mode()
def route_query(state: RAGState):
    enc = bert_tokenizer(state.question, truncation=True, max_length=64, return_tensors="pt").to(device)
    probs = bert(**enc).logits.softmax(-1)[0]
    i = int(probs.argmax())
    return ROUTES[i], float(probs[i])

In [ ]:
def bert_node(state: RAGState) -> dict:
    route, confidence = route_query(state)
    picked = route
    if confidence < THRESHOLD:
        route = "rag_retrieval"  
    print(f"Router picked: {picked} ({confidence:.2f}) -> using: {route}")
    return {"decision": route, "confidence": confidence}


def route_decision(state: RAGState) -> Literal["normal", "web", "RAG"]:
    if state.decision == "direct_answer":
        return "normal"
    if state.decision == "web_search":
        return "web"
    return "RAG"


grapher = StateGraph(RAGState)
grapher.add_node("bert", bert_node)
grapher.add_node("normal", llm_answer_node)
grapher.add_node("web", web_search_node)
grapher.add_node("web_generate", generate_node)   
grapher.add_node("RAG", graph_builder)           

grapher.set_entry_point("bert")
grapher.add_conditional_edges("bert", route_decision)
grapher.add_edge("normal", END)
grapher.add_edge("web", "web_generate")
grapher.add_edge("web_generate", END)
grapher.add_edge("RAG", END)

adaptive_rag = grapher.compile()

try:
    display(Image(adaptive_rag.get_graph().draw_mermaid_png(max_retries=3, retry_delay=2.0)))
except Exception:
    print(adaptive_rag.get_graph().draw_mermaid())  # mermaid.ink unreachable: show the diagram as text

In [ ]:
result = adaptive_rag.invoke({"question": "What are reflection tokens in Self-RAG?"})
print(result["decision"], result["confidence"])
print(result["generation"])


## Evaluation

Every number below is written to `results/` and summarised in the README.

In [ ]:

import json
import statistics
import time
from pathlib import Path

from langchain_core.callbacks import UsageMetadataCallbackHandler
from langsmith import traceable
from sklearn.metrics import classification_report, confusion_matrix

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
router_data = json.loads(Path("eval/router_dataset.json").read_text(encoding="utf-8"))


def timed(fn, *args):
    """Run fn(*args) and return (result, milliseconds), waiting for queued GPU work to finish."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    out = fn(*args)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return out, (time.perf_counter() - start) * 1000


def latency_stats(ms):
    ms = sorted(ms)
    return {
        "mean_ms": statistics.mean(ms),
        "median_ms": statistics.median(ms),
        "p95_ms": ms[round(0.95 * (len(ms) - 1))],
        "min_ms": ms[0],
        "max_ms": ms[-1],
        "total_s": sum(ms) / 1000,
    }


def evaluate_router(name, route_fn, token_fn, model, extra=None, warmup=3):
    """Accuracy, latency and token usage of a router on the held-out test queries."""
    test = router_data["test"]
    for row in test[:warmup]:
        route_fn(row["query"])
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    rows = []
    for row in test:
        (choice, confidence), ms = timed(route_fn, row["query"])
        rows.append({
            "query": row["query"],
            "label": row["label"],
            "predicted": choice,
            "confidence": confidence,
            "final_route": choice if confidence >= THRESHOLD else "rag_retrieval",
            "input_tokens": token_fn(row["query"]),
            "latency_ms": ms,
        })

    labels = router_data["labels"]
    y_true = [r["label"] for r in rows]
    y_pred = [r["predicted"] for r in rows]
    tokens = [r["input_tokens"] for r in rows]
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    summary = {
        "router": name,
        "n_test": len(rows),
        "accuracy": sum(t == p for t, p in zip(y_true, y_pred)) / len(rows),
        "accuracy_after_fallback": sum(r["label"] == r["final_route"] for r in rows) / len(rows),
        "threshold": THRESHOLD,
        "fallback_count": sum(r["confidence"] < THRESHOLD for r in rows),
        "mean_confidence": statistics.mean(r["confidence"] for r in rows),
        "macro_f1": report["macro avg"]["f1-score"],
        "per_class": {l: {k: report[l][k] for k in ("precision", "recall", "f1-score")} for l in labels},
        "confusion_matrix": {"labels": labels, "matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()},
        "latency": latency_stats([r["latency_ms"] for r in rows]),
        "tokens": {"mean_input_per_query": statistics.mean(tokens), "total_input": sum(tokens), "output_per_query": 0},
        "parameters": sum(p.numel() for p in model.parameters()),
        "devices": sorted({str(d) for d in getattr(model, "hf_device_map", {}).values()}) or [str(next(model.parameters()).device)],
        "peak_gpu_memory_mb": torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else None,
        **(extra or {}),
    }
    (RESULTS / f"router_{name.lower()}.json").write_text(json.dumps({"summary": summary, "rows": rows}, indent=2), encoding="utf-8")
    return summary, rows


def show_router(summary, rows):
    keys = ("accuracy", "accuracy_after_fallback", "fallback_count", "macro_f1", "latency", "tokens", "peak_gpu_memory_mb", "devices")
    print(json.dumps({k: summary[k] for k in keys}, indent=2))
    print("\nConfusion matrix (rows = true, columns = predicted):", summary["confusion_matrix"]["labels"])
    for label, line in zip(summary["confusion_matrix"]["labels"], summary["confusion_matrix"]["matrix"]):
        print(f"  {label:>14}: {line}")
    print("\nMisrouted queries:")
    for r in rows:
        if r["predicted"] != r["label"]:
            print(f'  {r["label"]:>14} -> {r["predicted"]:<14} ({r["confidence"]:.2f})  {r["query"]}')


### BERT router: accuracy, latency, tokens

Same 30 held-out queries, same warm-up and timing as the JEV evaluation. BERT's token cost is just the query
(plus `[CLS]`/`[SEP]`); it also generates **0 output tokens**.

In [ ]:

@traceable(name="bert_router")  # shows up in LangSmith when LANGSMITH_TRACING=true
def bert_route(query):
    return route_query(RAGState(question=query))


def bert_input_tokens(query):
    return len(bert_tokenizer(query, truncation=True, max_length=64)["input_ids"])


bert_summary, bert_rows = evaluate_router("BERT", bert_route, bert_input_tokens, bert, extra={
    "model": BERT_MODEL, "load_s": BERT_LOAD_S, "train_s": BERT_TRAIN_S, "training_examples": len(router_train), "epochs": EPOCHS,
})
show_router(bert_summary, bert_rows)


### JEV vs BERT

Reads `results/router_jev.json` (written by `adaptive.ipynb`). The RAG part is identical in both notebooks, so RAGAS
runs once, in `adaptive.ipynb`.

In [ ]:

import pandas as pd

jev_file = RESULTS / "router_jev.json"
if jev_file.exists():
    jev = json.loads(jev_file.read_text(encoding="utf-8"))["summary"]
    compare = pd.DataFrame({
        s["router"]: {
            "accuracy": s["accuracy"],
            "accuracy after 0.7 fallback": s["accuracy_after_fallback"],
            "macro F1": s["macro_f1"],
            "fallbacks (conf < 0.7)": s["fallback_count"],
            "mean latency (ms)": s["latency"]["mean_ms"],
            "median latency (ms)": s["latency"]["median_ms"],
            "p95 latency (ms)": s["latency"]["p95_ms"],
            "mean input tokens / query": s["tokens"]["mean_input_per_query"],
            "total input tokens (30 queries)": s["tokens"]["total_input"],
            "output tokens / query": s["tokens"]["output_per_query"],
            "parameters (M)": s["parameters"] / 1e6,
            "peak GPU memory (MB)": s["peak_gpu_memory_mb"],
            "load time (s)": s["load_s"],
            "training time (s)": s.get("train_s", 0.0),
            "labeled training examples": s["training_examples"],
        }
        for s in (jev, bert_summary)
    })
    display(compare.round(3))
else:
    print("Run the evaluation section of adaptive.ipynb first to create results/router_jev.json")
